

# Query Augmentation in Context Engineering: A Principal-Level Technical Report

## Deterministic Query Transformation, Expansion, Decomposition, and Agentic Query Orchestration for Production-Grade Retrieval Pipelines

---

## 1. Problem Statement and Formal Framing

In any agentic AI system, the **query interface** constitutes the singular point of maximum information loss. The raw user utterance $q_{\text{raw}}$ undergoes an entropy-increasing transformation as it passes through retrieval, ranking, and generation stages. If intent is misidentified at ingress, no downstream component—regardless of sophistication—can recover the signal. This is the **query bottleneck theorem** in context engineering:

$$
\mathcal{L}_{\text{system}} \geq \mathcal{L}_{\text{query}} + \mathcal{L}_{\text{retrieval}} + \mathcal{L}_{\text{generation}}
$$

where $\mathcal{L}_{\text{query}}$ represents the information loss at the query stage. Crucially:

$$
\frac{\partial \mathcal{L}_{\text{system}}}{\partial \mathcal{L}_{\text{query}}} \gg \frac{\partial \mathcal{L}_{\text{system}}}{\partial \mathcal{L}_{\text{retrieval}}} \gg \frac{\partial \mathcal{L}_{\text{system}}}{\partial \mathcal{L}_{\text{generation}}}
$$

This establishes that **marginal improvements at the query stage yield disproportionately higher system-level accuracy gains** than equivalent effort applied at retrieval or generation. The formal objective of query augmentation is therefore:

$$
q^{*} = \arg\max_{q' \in \mathcal{Q}'} \; \mathbb{E}\left[\text{Recall}@k\left(\mathcal{R}(q'), \mathcal{D}_{\text{relevant}}\right)\right] \cdot \mathbb{E}\left[\text{Precision}@k\left(\mathcal{R}(q'), \mathcal{D}_{\text{relevant}}\right)\right]
$$

subject to:

$$
\text{latency}(q') \leq \tau_{\text{budget}}, \quad \text{tokens}(q') \leq T_{\text{max}}, \quad \text{drift}(q', q_{\text{raw}}) \leq \delta_{\text{max}}
$$

where $\mathcal{R}$ is the retrieval function, $\mathcal{D}_{\text{relevant}}$ is the ground-truth relevant document set, $\tau_{\text{budget}}$ is the latency SLA, $T_{\text{max}}$ is the token budget for augmented queries, and $\delta_{\text{max}}$ is the maximum permissible semantic drift from the user's original intent.

---

## 2. Dual-Pipeline Incompatibility Problem

A critical architectural constraint that necessitates query augmentation is the **dual-pipeline incompatibility**: a query optimized for one downstream consumer is suboptimal—often catastrophically so—for another.

### 2.1 Formal Characterization

Let $f_{\text{vec}}: \mathcal{Q} \to \mathbb{R}^d$ be the embedding function for vector retrieval, and let $f_{\text{LLM}}: \mathcal{Q} \to \mathcal{A}$ be the LLM generation function. The optimal query forms diverge:

$$
q^{*}_{\text{retrieval}} = \arg\max_{q'} \; \text{sim}\left(f_{\text{vec}}(q'), f_{\text{vec}}(d^{*})\right), \quad d^{*} \in \mathcal{D}_{\text{relevant}}
$$

$$
q^{*}_{\text{generation}} = \arg\max_{q'} \; P_{\text{LLM}}\left(a^{*} \mid q', \mathcal{C}\right), \quad a^{*} \in \mathcal{A}_{\text{correct}}
$$

These two objectives produce **structurally incompatible** query forms:

| Property | Retrieval-Optimal Query | Generation-Optimal Query |
|---|---|---|
| **Lexical Density** | High keyword density, technical terminology | Natural language with contextual framing |
| **Disambiguation** | Explicit entity references, schema-aligned filters | Conversational co-reference resolution |
| **Structure** | Short, focused, mono-topical | Extended, multi-faceted, with constraints |
| **Token Profile** | Compact embedding-friendly surface | Rich instruction-following preamble |

This incompatibility mandates that query augmentation produce **multiple typed query representations** routed to appropriate consumers, not a single rewritten string.

### 2.2 Typed Query Envelope

Define the augmented query as a typed envelope:

```protobuf
message AugmentedQueryEnvelope {
  string request_id = 1;
  string raw_query = 2;
  QueryIntent intent = 3;
  repeated RetrievalQuery retrieval_queries = 4;
  GenerationQuery generation_query = 5;
  repeated FilterPredicate metadata_filters = 6;
  repeated string target_collections = 7;
  float drift_score = 8;
  int64 deadline_ms = 9;
  QueryProvenance provenance = 10;
}

message RetrievalQuery {
  string query_text = 1;
  QueryType type = 2;  // SEMANTIC, KEYWORD, HYBRID, GRAPH
  float weight = 3;
  repeated string target_fields = 4;
  int32 top_k = 5;
}

message GenerationQuery {
  string rewritten_query = 1;
  repeated string decomposed_subqueries = 2;
  string synthesized_intent = 3;
}

message QueryProvenance {
  string transformation_method = 1;
  string model_id = 2;
  float confidence = 3;
  int64 latency_ms = 4;
}
```

This protocol ensures every downstream consumer receives a query artifact in its optimal representation, with provenance tracking for observability and debugging.

---

## 3. Query Rewriting: Intent-Preserving Transformation

### 3.1 Formal Definition

Query rewriting is a **surjective mapping** $\phi: \mathcal{Q}_{\text{raw}} \to \mathcal{Q}_{\text{canonical}}$ that transforms an ill-formed, ambiguous, or suboptimally phrased user query into a canonical form that maximizes downstream retrieval precision while preserving semantic intent:

$$
\phi(q_{\text{raw}}) = q_{\text{rewritten}} \quad \text{s.t.} \quad \text{intent}(q_{\text{rewritten}}) \equiv \text{intent}(q_{\text{raw}}) \;\wedge\; \text{Recall}(\mathcal{R}(q_{\text{rewritten}})) \geq \text{Recall}(\mathcal{R}(q_{\text{raw}}))
$$

### 3.2 SOTA Rewriting Architecture: Step-Back Prompting with Chain-of-Density Compression

The state-of-the-art approach combines **Step-Back Prompting** (Zheng et al., 2023) with **Chain-of-Density** compression to produce maximally informative, retrieval-optimized rewrites.

**Step-Back Abstraction.** Instead of rewriting at the surface level, the model first generates a more abstract "step-back" question that captures the underlying principle, then produces a concrete retrieval query:

$$
q_{\text{abstract}} = \text{LLM}_{\text{step-back}}(q_{\text{raw}}), \quad q_{\text{rewritten}} = \text{LLM}_{\text{concretize}}(q_{\text{abstract}}, q_{\text{raw}})
$$

**Chain-of-Density Compression.** The rewritten query is iteratively densified: each pass increases entity density while maintaining fixed token length, ensuring maximum information per token:

$$
q^{(t+1)} = \text{Densify}\left(q^{(t)}\right) \quad \text{s.t.} \quad |q^{(t+1)}|_{\text{tokens}} \leq |q^{(t)}|_{\text{tokens}} \;\wedge\; \text{entities}(q^{(t+1)}) \geq \text{entities}(q^{(t)})
$$

### 3.3 Pseudo-Algorithm: Production Query Rewriter

```
ALGORITHM: SOTA_QUERY_REWRITE
──────────────────────────────────────────────────────────────

Input:
  q_raw         : string        — raw user utterance
  history       : ConvTurn[]    — session conversation history (last N turns)
  schema        : CollectionSchema — target data schema for filter extraction
  τ_max         : int           — latency budget in milliseconds
  T_budget      : int           — token budget for rewritten query

Output:
  envelope      : AugmentedQueryEnvelope

Procedure:

  1. CO-REFERENCE RESOLUTION
     ─────────────────────────
     q_resolved ← RESOLVE_COREFERENCES(q_raw, history)
     // Replace pronouns, ellipsis, implicit references
     // with explicit entity mentions from conversation history.
     //
     // Method: Sliding-window attention over last K turns.
     // For each anaphoric expression in q_raw, find the
     // most recent antecedent in history using entity salience scoring:
     //
     //   antecedent* = argmax_{e ∈ entities(history)}
     //                   [ salience(e) · recency_decay(e) · type_match(e, anaphor) ]
     //
     //   salience(e) = tf(e, history) · idf(e, corpus) · positional_weight(e)
     //   recency_decay(e) = exp(-λ · (current_turn - last_mention_turn(e)))

  2. INTENT CLASSIFICATION
     ──────────────────────
     intent ← CLASSIFY_INTENT(q_resolved)
     // Typed intent taxonomy:
     //   FACTUAL_LOOKUP | PROCEDURAL | COMPARATIVE | DIAGNOSTIC |
     //   AGGREGATION | EXPLORATORY | TRANSACTIONAL | CLARIFICATION
     //
     // Use a fine-tuned classifier or structured LLM extraction:
     //   intent = LLM(
     //     system="Classify the query intent into exactly one of: {taxonomy}",
     //     user=q_resolved,
     //     response_format={"type": "json", "schema": IntentSchema}
     //   )
     //
     // Confidence gating: if P(intent) < θ_intent (e.g., 0.7),
     // set intent = AMBIGUOUS and flag for clarification request.

  3. STEP-BACK ABSTRACTION
     ──────────────────────
     q_abstract ← STEP_BACK(q_resolved, intent)
     // Generate a higher-level conceptual question that captures
     // the underlying principle or domain area:
     //
     //   q_abstract = LLM(
     //     system="Given the user's specific question, generate a more
     //             general 'step-back' question that identifies the
     //             underlying concept, principle, or domain area.",
     //     user=q_resolved
     //   )
     //
     // Example:
     //   q_resolved:  "Why does my API call return 401 after token refresh?"
     //   q_abstract:  "How does OAuth2 token refresh interact with
     //                 authentication header validation in REST APIs?"

  4. ENTITY AND FILTER EXTRACTION
     ────────────────────────────
     entities ← EXTRACT_ENTITIES(q_resolved, schema)
     filters  ← DERIVE_FILTERS(entities, schema)
     // Extract named entities, technical terms, version numbers,
     // date ranges, and schema-aligned filter predicates.
     //
     // For each extracted entity e:
     //   IF e matches schema.field[f].type AND schema.field[f].filterable:
     //     filters.append(FilterPredicate(field=f, op=EQ|RANGE|IN, value=e))
     //
     // Example:
     //   q_resolved: "errors in payment service after v2.3 deployment last week"
     //   entities:   {service: "payment", version: "2.3", time: "last_week"}
     //   filters:    [{field: "service", op: EQ, val: "payment"},
     //                {field: "version", op: GTE, val: "2.3"},
     //                {field: "timestamp", op: GTE, val: NOW - 7d}]

  5. KEYWORD DENSIFICATION (Chain-of-Density)
     ─────────────────────────────────────────
     q_dense ← q_resolved
     FOR t = 1 TO N_density_passes (typically 2-3):
       q_dense ← DENSIFY(q_dense, entities, intent)
       // Each pass:
       //   - Identify missing salient entities from the domain
       //   - Replace vague terms with precise technical vocabulary
       //   - Inject synonyms and acronym expansions
       //   - Maintain or reduce token count
       //
       // Densification score:
       //   D(q) = |unique_entities(q)| / |tokens(q)|
       //
       // Terminate when D(q^(t+1)) - D(q^(t)) < ε_density

  6. MULTI-REPRESENTATION SYNTHESIS
     ──────────────────────────────
     // Produce distinct query forms for each retrieval modality:

     q_semantic  ← SYNTHESIZE_SEMANTIC(q_abstract, q_dense)
     // Natural-language query optimized for dense embedding similarity.
     // Emphasize conceptual coverage over keyword precision.

     q_keyword   ← EXTRACT_KEYWORDS(q_dense, top_k=10)
     // BM25-optimized keyword set with TF-IDF weighting.
     // Apply domain-specific stop-word removal and stemming.

     q_graph     ← CONSTRUCT_GRAPH_QUERY(entities, schema)
     // If graph/knowledge-base retrieval is available:
     // Generate SPARQL, Cypher, or structured traversal query
     // targeting entity relationships.

     q_generation ← SYNTHESIZE_GENERATION(q_resolved, q_abstract, intent)
     // Full natural-language question optimized for LLM comprehension.
     // Include explicit constraints, expected answer format,
     // and disambiguation context.

  7. DRIFT VALIDATION
     ─────────────────
     drift ← COMPUTE_DRIFT(q_raw, q_semantic, q_generation)
     // Drift metric: cosine distance in embedding space plus
     // entity overlap penalty:
     //
     //   drift(q, q') = α · (1 - cos(embed(q), embed(q')))
     //                + β · (1 - |entities(q) ∩ entities(q')| / |entities(q)|)
     //
     //   where α + β = 1, typically α=0.6, β=0.4
     //
     // IF drift > δ_max:
     //   FALL_BACK to q_resolved (minimally processed) for affected representations
     //   LOG drift_violation event with full provenance

  8. ENVELOPE ASSEMBLY
     ──────────────────
     envelope ← AugmentedQueryEnvelope(
       request_id   = generate_uuid(),
       raw_query    = q_raw,
       intent       = intent,
       retrieval_queries = [
         RetrievalQuery(q_semantic, SEMANTIC, weight=0.6, top_k=20),
         RetrievalQuery(q_keyword,  KEYWORD,  weight=0.3, top_k=30),
         RetrievalQuery(q_graph,    GRAPH,    weight=0.1, top_k=10)
       ],
       generation_query = GenerationQuery(q_generation, [], q_abstract),
       metadata_filters = filters,
       target_collections = ROUTE_COLLECTIONS(intent, entities, schema),
       drift_score  = drift,
       deadline_ms  = τ_max - elapsed_ms(),
       provenance   = QueryProvenance("SOTA_REWRITE_v2", model_id, confidence, elapsed)
     )

  RETURN envelope
```

### 3.4 Drift Control: Formal Guarantee

The drift constraint is non-negotiable in production. Define the **Intent Preservation Index (IPI)**:

$$
\text{IPI}(q_{\text{raw}}, q') = \frac{\sum_{e \in \mathcal{E}(q_{\text{raw}})} \mathbb{1}[e \in \mathcal{E}(q')]}{\left|\mathcal{E}(q_{\text{raw}})\right|} \cdot \cos\left(\mathbf{v}(q_{\text{raw}}), \mathbf{v}(q')\right)
$$

where $\mathcal{E}(\cdot)$ extracts salient entities and $\mathbf{v}(\cdot)$ is the dense embedding. A rewritten query is accepted only if:

$$
\text{IPI}(q_{\text{raw}}, q') \geq \theta_{\text{IPI}} \quad (\text{typically } \theta_{\text{IPI}} = 0.85)
$$

If this threshold is violated, the system falls back to the co-reference-resolved form $q_{\text{resolved}}$ and emits a structured observability event.

---

## 4. Query Expansion: Controlled Multi-Query Generation

### 4.1 Formal Definition

Query expansion generates a set of $n$ related queries $\{q_1', q_2', \ldots, q_n'\}$ from a single input $q$ to increase recall coverage across the retrieval corpus:

$$
\mathcal{Q}_{\text{expanded}} = \text{Expand}(q) = \{q_i' \mid i \in [1, n], \; \text{sim}_{\text{intent}}(q_i', q) \geq \theta_{\text{sim}}\}
$$

The aggregated retrieval result is:

$$
\mathcal{D}_{\text{expanded}} = \bigcup_{i=1}^{n} \mathcal{R}(q_i', k_i) \quad \text{with deduplication and re-ranking}
$$

### 4.2 SOTA Expansion: Hypothetical Document Embeddings (HyDE) + Reciprocal Rank Fusion

**HyDE (Gao et al., 2022).** Instead of embedding the query directly, generate a hypothetical answer document $\hat{d}$, then use $\hat{d}$'s embedding for retrieval:

$$
\hat{d} = \text{LLM}(q), \quad \mathbf{v}_{\text{retrieval}} = f_{\text{vec}}(\hat{d})
$$

This bridges the **query-document embedding gap**: the hypothetical document occupies the same embedding subspace as real documents, whereas queries occupy a different distributional region.

**Multi-Perspective Expansion.** Generate expansions along orthogonal semantic axes:

$$
q_i' = \text{LLM}\left(q, \text{perspective}_i\right), \quad \text{perspective}_i \in \{\text{synonym}, \text{hypernym}, \text{related\_concept}, \text{contrastive}, \text{procedural}\}
$$

**Reciprocal Rank Fusion (RRF).** Aggregate results from all expanded queries using RRF to avoid any single expansion dominating:

$$
\text{RRF\_score}(d) = \sum_{i=1}^{n} \frac{1}{k + \text{rank}_i(d)}
$$

where $k$ is a smoothing constant (typically $k = 60$), and $\text{rank}_i(d)$ is the rank of document $d$ in the result set of query $q_i'$.

### 4.3 Pseudo-Algorithm: Production Query Expansion

```
ALGORITHM: CONTROLLED_QUERY_EXPANSION
──────────────────────────────────────────────────────────────

Input:
  q_rewritten   : string           — rewritten query from Stage 3
  intent        : QueryIntent      — classified intent
  n_max         : int              — maximum expansion count (default: 5)
  τ_budget      : int              — remaining latency budget (ms)
  retrieval_mode: enum             — {SEMANTIC, KEYWORD, HYBRID}

Output:
  expanded_set  : RetrievalQuery[] — weighted set of retrieval queries
  rrf_config    : RRFConfig        — fusion parameters

Procedure:

  1. EXPANSION STRATEGY SELECTION
     ────────────────────────────
     strategy ← SELECT_STRATEGY(intent, retrieval_mode)
     //
     // Strategy matrix (intent × retrieval_mode → expansion_type[]):
     //
     // ┌──────────────────┬──────────────────────────────────────────┐
     // │ Intent           │ Expansion Types                          │
     // ├──────────────────┼──────────────────────────────────────────┤
     // │ FACTUAL_LOOKUP   │ [SYNONYM, ACRONYM_EXPAND, HyDE]         │
     // │ PROCEDURAL       │ [STEP_ENUMERATE, HyDE, RELATED_CONCEPT] │
     // │ DIAGNOSTIC       │ [SYMPTOM_EXPAND, CAUSAL, CONTRASTIVE]   │
     // │ COMPARATIVE      │ [ENTITY_PAIR, ATTRIBUTE_AXIS, HyDE]     │
     // │ AGGREGATION      │ [FACET_ENUMERATE, TEMPORAL_WINDOW]      │
     // │ EXPLORATORY      │ [HYPERNYM, RELATED_CONCEPT, MULTI_ANGLE]│
     // └──────────────────┴──────────────────────────────────────────┘

  2. HYPOTHETICAL DOCUMENT GENERATION (HyDE)
     ────────────────────────────────────────
     IF HyDE ∈ strategy:
       d_hyp ← LLM(
         system="Given the following question, write a short paragraph that
                 would be found in a document that perfectly answers it.
                 Do not answer the question—write the SOURCE document passage.",
         user=q_rewritten,
         max_tokens=150,
         temperature=0.3
       )
       q_hyde ← RetrievalQuery(
         query_text=d_hyp, type=SEMANTIC, weight=0.4, top_k=15
       )

  3. MULTI-PERSPECTIVE EXPANSION
     ───────────────────────────
     perspectives ← strategy \ {HyDE}
     expanded_queries ← []

     FOR EACH perspective_type IN perspectives:
       q_exp ← LLM(
         system="Rewrite the following query from the perspective of
                 {perspective_type}. Produce a single, focused query
                 that would retrieve documents relevant to this aspect.
                 Output ONLY the rewritten query, nothing else.",
         user=q_rewritten,
         max_tokens=60,
         temperature=0.5
       )

       // Compute intent-preservation score:
       sim ← cosine(embed(q_rewritten), embed(q_exp))

       IF sim ≥ θ_expansion (typically 0.65):
         expanded_queries.append(
           RetrievalQuery(q_exp, SEMANTIC, weight=sim * 0.3, top_k=10)
         )

     // BUDGET ENFORCEMENT: if |expanded_queries| > n_max - 1:
     //   Sort by weight descending, truncate to n_max - 1.

  4. DIVERSITY-WEIGHTED DEDUPLICATION
     ────────────────────────────────
     // Remove near-duplicate expansions via MMR-style filtering:
     //
     //   MMR(q_i) = λ · sim(q_i, q_rewritten) - (1-λ) · max_{q_j ∈ S} sim(q_i, q_j)
     //
     //   where S is the already-selected set, λ = 0.7
     //
     // Greedily select queries maximizing MMR until |S| = n_max.

     expanded_set ← MMR_SELECT(
       candidates = [q_hyde] + expanded_queries + [original_as_retrieval(q_rewritten)],
       λ = 0.7,
       n = n_max
     )

  5. WEIGHT NORMALIZATION
     ────────────────────
     // Normalize weights to sum to 1.0 for RRF scoring:
     total_w ← SUM(q.weight for q in expanded_set)
     FOR EACH q IN expanded_set:
       q.weight ← q.weight / total_w

  6. RRF CONFIGURATION
     ──────────────────
     rrf_config ← RRFConfig(
       k = 60,
       score_function = "reciprocal_rank",
       dedup_field = "document_id",
       min_appearances = 1,     // document must appear in ≥1 result set
       max_results = top_k_final
     )

  RETURN expanded_set, rrf_config
```

### 4.4 Controlling Query Drift in Expansion

The three pathological modes of expansion failure—**drift**, **over-expansion**, and **latency blowup**—are controlled mechanically:

**Drift Control.** Each expanded query $q_i'$ is validated against the original via the pairwise similarity gate:

$$
q_i' \in \mathcal{Q}_{\text{accepted}} \iff \cos\left(\mathbf{v}(q_i'), \mathbf{v}(q_{\text{raw}})\right) \geq \theta_{\text{expansion}}
$$

**Over-Expansion Control.** Apply Maximal Marginal Relevance (MMR) selection to enforce diversity within the expansion set while bounding cardinality:

$$
\text{MMR}(q_i) = \lambda \cdot \text{sim}(q_i, q_{\text{raw}}) - (1 - \lambda) \cdot \max_{q_j \in \mathcal{S}} \text{sim}(q_i, q_j)
$$

where $\mathcal{S}$ is the already-selected expansion set and $\lambda \in [0.6, 0.8]$.

**Latency Control.** Execute expanded queries in parallel with a fan-out ceiling $n_{\text{max}}$ and a shared deadline. The RRF aggregation step has $O(n \cdot k \log k)$ complexity, which is negligible compared to retrieval latency:

$$
\text{latency}_{\text{expansion}} = \max_{i \in [1, n]} \text{latency}(\mathcal{R}(q_i')) + \text{latency}_{\text{RRF}} \approx \text{latency}_{\text{single\_retrieval}} + O(1)
$$

provided queries execute in parallel with proper fan-out and deadline propagation.

---

## 5. Query Decomposition: Divide-Retrieve-Synthesize

### 5.1 Formal Definition

For a complex multi-faceted query $q$ that requires information from $m$ distinct knowledge domains or document clusters, decomposition produces:

$$
\text{Decompose}(q) = \{s_1, s_2, \ldots, s_m\} \quad \text{s.t.} \quad \bigcup_{j=1}^{m} \text{scope}(s_j) \supseteq \text{scope}(q) \;\wedge\; \forall j \neq k: |\text{scope}(s_j) \cap \text{scope}(s_k)| \leq \epsilon
$$

where each sub-query $s_j$ targets a single information facet, the union of sub-query scopes covers the original query, and overlap between sub-queries is minimized.

### 5.2 SOTA Decomposition: Least-to-Most with Dependency DAG

The state-of-the-art decomposition approach constructs a **Directed Acyclic Graph (DAG)** of sub-queries with explicit dependency edges, enabling:

1. **Parallelization** of independent sub-queries
2. **Sequential chaining** where one sub-query's result informs another
3. **Early termination** if a critical sub-query yields no results

**Dependency DAG Construction.** For a decomposed set $\{s_1, \ldots, s_m\}$, define the dependency relation:

$$
s_i \to s_j \iff \text{answer}(s_i) \text{ is required to formulate or interpret } s_j
$$

The execution schedule is the topological sort of this DAG, with independent nodes executed in parallel:

$$
\text{Schedule} = \text{TopoSort}\left(\mathcal{G} = (\{s_1, \ldots, s_m\}, \{(s_i, s_j) \mid s_i \to s_j\})\right)
$$

### 5.3 Pseudo-Algorithm: Production Query Decomposition

```
ALGORITHM: DAG_QUERY_DECOMPOSITION
──────────────────────────────────────────────────────────────

Input:
  q_rewritten   : string          — rewritten query
  intent        : QueryIntent     — classified intent
  schema        : CollectionSchema
  max_depth     : int             — maximum decomposition depth (default: 3)
  max_subqueries: int             — maximum sub-query count (default: 6)

Output:
  dag           : QueryDAG        — dependency graph of sub-queries
  schedule      : ExecutionSchedule — topologically sorted execution plan

Procedure:

  1. COMPLEXITY ASSESSMENT
     ─────────────────────
     complexity ← ASSESS_COMPLEXITY(q_rewritten)
     //
     // Complexity scoring function:
     //   C(q) = w_1 · |entities(q)|
     //        + w_2 · |relation_types(q)|
     //        + w_3 · |temporal_references(q)|
     //        + w_4 · |comparison_axes(q)|
     //        + w_5 · conditional_depth(q)
     //
     //   where w = [0.25, 0.25, 0.15, 0.2, 0.15]
     //
     // IF C(q) < θ_decompose (e.g., 2.0):
     //   SKIP decomposition; return single-node DAG.
     //   Rationale: simple queries lose precision through decomposition.

     IF complexity < θ_decompose:
       dag ← SingleNodeDAG(q_rewritten)
       schedule ← [q_rewritten]
       RETURN dag, schedule

  2. FACET EXTRACTION
     ─────────────────
     facets ← EXTRACT_FACETS(q_rewritten, intent)
     //
     // Use structured extraction to identify distinct information needs:
     //
     //   facets = LLM(
     //     system="Analyze this complex question and identify the distinct
     //             information facets that need to be answered independently.
     //             For each facet, provide:
     //             - facet_id: unique identifier
     //             - sub_question: focused question for this facet
     //             - required_info_type: {factual, procedural, comparative, temporal}
     //             - depends_on: list of facet_ids whose answers are needed first
     //             - target_collection_hint: which data source likely contains this",
     //     user=q_rewritten,
     //     response_format={"type": "json", "schema": FacetListSchema}
     //   )
     //
     // CARDINALITY ENFORCEMENT:
     //   IF |facets| > max_subqueries:
     //     Merge the two most similar facets (by embedding distance) iteratively
     //     until |facets| ≤ max_subqueries.

  3. DEPENDENCY DAG CONSTRUCTION
     ───────────────────────────
     dag ← DirectedAcyclicGraph()
     FOR EACH facet IN facets:
       node ← SubQueryNode(
         id = facet.facet_id,
         query = facet.sub_question,
         info_type = facet.required_info_type,
         collection_hint = facet.target_collection_hint,
         status = PENDING
       )
       dag.add_node(node)

     FOR EACH facet IN facets:
       FOR EACH dep_id IN facet.depends_on:
         dag.add_edge(dep_id → facet.facet_id)

     // CYCLE DETECTION AND REPAIR:
     IF dag.has_cycle():
       // Break cycles by removing the edge with lowest confidence score:
       back_edges ← dag.find_back_edges()
       FOR EACH edge IN back_edges:
         dag.remove_edge(edge)
         IF NOT dag.has_cycle():
           BREAK

     // DEPTH ENFORCEMENT:
     IF dag.longest_path() > max_depth:
       // Flatten: promote deep nodes to have dependencies only on root nodes
       dag.flatten_to_depth(max_depth)

  4. EXECUTION SCHEDULE GENERATION
     ─────────────────────────────
     schedule ← TopologicalSort(dag)
     //
     // Group into execution waves (nodes at same depth execute in parallel):
     //
     //   Wave 0: all root nodes (no dependencies)
     //   Wave 1: nodes depending only on Wave 0 nodes
     //   ...
     //   Wave d: nodes depending on Wave d-1 or earlier
     //
     // Each wave has a timeout = τ_budget / (max_depth + 1)

     waves ← []
     FOR depth = 0 TO dag.max_depth():
       wave ← dag.nodes_at_depth(depth)
       waves.append(ExecutionWave(
         nodes = wave,
         timeout_ms = τ_budget / (dag.max_depth() + 1),
         parallel = TRUE
       ))

  5. SUB-QUERY REFINEMENT WITH CHAINED CONTEXT
     ──────────────────────────────────────────
     // After Wave i completes, inject retrieved results into
     // dependent sub-queries in Wave i+1:
     //
     // FOR EACH node IN wave[i+1]:
     //   dep_results ← GATHER(node.dependencies, results_cache)
     //   node.query ← REFINE_WITH_CONTEXT(node.query, dep_results)
     //
     // This enables progressive refinement:
     //   "What is the latency of service X?" (Wave 0)
     //   → result: "p99 latency is 450ms"
     //   "What caused the latency spike in service X beyond 450ms
     //    last Tuesday?" (Wave 1, refined with Wave 0 result)

  6. SYNTHESIS STRATEGY ASSIGNMENT
     ─────────────────────────────
     dag.synthesis_strategy ← SELECT_SYNTHESIS(intent, |facets|)
     //
     // Synthesis strategies:
     //   CONCATENATIVE:     Join all sub-results with section headers
     //   COMPARATIVE_TABLE: Format as comparison matrix
     //   NARRATIVE_MERGE:   LLM synthesizes a coherent narrative
     //   HIERARCHICAL:      Organize by DAG structure with nesting
     //
     // The synthesis strategy is passed to the generation phase
     // as part of the GenerationQuery in the AugmentedQueryEnvelope.

  RETURN dag, schedule
```

### 5.4 Synthesis Aggregation: Formal Fusion

After all sub-queries complete, results must be fused into a coherent response. Define the synthesis function:

$$
\text{Synthesize}(\{(s_j, \mathcal{D}_j)\}_{j=1}^m, q_{\text{original}}) = \text{LLM}\left(q_{\text{original}}, \bigoplus_{j=1}^{m} \text{Compress}(s_j, \mathcal{D}_j)\right)
$$

where $\bigoplus$ is the structured concatenation operator and $\text{Compress}$ extracts the minimal sufficient evidence from each sub-query's retrieval results:

$$
\text{Compress}(s_j, \mathcal{D}_j) = \arg\min_{c} |c|_{\text{tokens}} \quad \text{s.t.} \quad \text{NLI}(c \models s_j) \geq \theta_{\text{entailment}}
$$

This ensures each sub-result is compressed to its minimal entailing passage before injection into the synthesis context window.

---

## 6. Query Agents: Autonomous Query Orchestration

### 6.1 Formal Definition

A **Query Agent** is a bounded control loop $\mathcal{A}_q$ that autonomously selects and composes query augmentation strategies—rewriting, expansion, decomposition, filter construction, collection routing, and result evaluation—based on runtime analysis of the query, available data schema, and intermediate retrieval results:

$$
\mathcal{A}_q: (q_{\text{raw}}, \mathcal{S}, \mathcal{H}, \mathcal{C}_{\text{schema}}) \to \text{AugmentedQueryEnvelope}
$$

where $\mathcal{S}$ is the available tool/collection set, $\mathcal{H}$ is conversation history, and $\mathcal{C}_{\text{schema}}$ is the collection schema registry.

### 6.2 Agent Architecture: ReAct with Typed Tool Protocol

The query agent implements a **ReAct (Reason + Act)** loop augmented with typed tool contracts over MCP:

```
┌─────────────────────────────────────────────────────────┐
│                    QUERY AGENT LOOP                      │
│                                                          │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐             │
│  │ ANALYZE  │──▶│  PLAN    │──▶│  ROUTE   │             │
│  │ (intent, │   │ (select  │   │ (choose  │             │
│  │  schema, │   │  strategy│   │  collect-│             │
│  │  complex)│   │  stack)  │   │  ions)   │             │
│  └──────────┘   └──────────┘   └────┬─────┘             │
│                                      │                   │
│              ┌───────────────────────▼──────────┐        │
│              │         EXECUTE QUERIES           │        │
│              │  (parallel across collections,    │        │
│              │   mixed search + aggregation)     │        │
│              └───────────────────────┬──────────┘        │
│                                      │                   │
│  ┌──────────┐   ┌──────────┐   ┌────▼─────┐             │
│  │ RESPOND  │◀──│  MERGE   │◀──│ EVALUATE │             │
│  │ (synthe- │   │  (RRF +  │   │ (suffic- │◀──┐        │
│  │  size)   │   │  dedup)  │   │  iency?) │   │        │
│  └──────────┘   └──────────┘   └────┬─────┘   │        │
│                                      │ NO      │        │
│                                      ▼         │        │
│                               ┌──────────┐     │        │
│                               │ RE-QUERY │─────┘        │
│                               │ (adjust  │              │
│                               │  filters,│              │
│                               │  expand) │              │
│                               └──────────┘              │
│                                                          │
│  Bounded by: max_iterations, deadline_ms, token_budget   │
└─────────────────────────────────────────────────────────┘
```

### 6.3 Pseudo-Algorithm: Production Query Agent

```
ALGORITHM: QUERY_AGENT_ORCHESTRATOR
──────────────────────────────────────────────────────────────

Input:
  q_raw           : string             — raw user utterance
  history         : ConvTurn[]         — conversation history
  collection_registry : CollectionMeta[] — schema, stats, capabilities per collection
  tool_registry   : ToolManifest[]     — available MCP tool servers
  config          : AgentConfig        — max_iterations, deadline_ms, token_budget,
                                          sufficiency_threshold, min_results

Output:
  response        : AgentResponse      — final answer + provenance + traces

Procedure:

  iteration ← 0
  results_cache ← {}
  action_trace ← []
  remaining_budget_ms ← config.deadline_ms

  1. INITIAL ANALYSIS
     ────────────────
     analysis ← ANALYZE(q_raw, history, collection_registry)
     //
     // Structured analysis output:
     //
     //   analysis = LLM(
     //     system="You are a query planning agent. Analyze the user query
     //             given the available data collections and conversation context.
     //             Determine:
     //             1. primary_intent: the user's core information need
     //             2. required_operations: [SEARCH | AGGREGATE | FILTER | COMPARE]
     //             3. target_collections: ranked list of relevant collections
     //                with justification (matched by schema fields, data types)
     //             4. complexity_class: SIMPLE | MULTI_FACET | TEMPORAL | COMPARATIVE
     //             5. required_augmentations: [REWRITE | EXPAND | DECOMPOSE | NONE]
     //             6. initial_filters: extracted filter predicates from the query
     //             7. confidence: float in [0,1]",
     //     user=FORMAT(q_raw, history, collection_registry.summaries),
     //     response_format=AnalysisSchema
     //   )
     //
     // Time cost: ~200-500ms (single LLM call)
     // Token cost: ~800-1200 input, ~200-400 output

     action_trace.append(TraceEntry("ANALYZE", analysis, elapsed_ms()))

  2. STRATEGY PLANNING
     ──────────────────
     plan ← PLAN_STRATEGY(analysis, config)
     //
     // Decision matrix for strategy selection:
     //
     //   IF analysis.complexity_class == SIMPLE:
     //     plan.augmentation = [REWRITE]
     //     plan.execution = SINGLE_COLLECTION_SEARCH
     //     plan.max_iterations = 2
     //
     //   ELIF analysis.complexity_class == MULTI_FACET:
     //     plan.augmentation = [REWRITE, DECOMPOSE]
     //     plan.execution = MULTI_COLLECTION_PARALLEL
     //     plan.max_iterations = 3
     //
     //   ELIF analysis.complexity_class == TEMPORAL:
     //     plan.augmentation = [REWRITE, FILTER_INJECT]
     //     plan.execution = TIME_WINDOWED_SEARCH
     //     plan.max_iterations = 2
     //
     //   ELIF analysis.complexity_class == COMPARATIVE:
     //     plan.augmentation = [REWRITE, DECOMPOSE, EXPAND]
     //     plan.execution = MULTI_ENTITY_PARALLEL
     //     plan.max_iterations = 3
     //
     // Each plan specifies:
     //   - ordered augmentation pipeline
     //   - target collection routing
     //   - search type per collection (vector / keyword / hybrid / aggregation)
     //   - merge strategy (RRF / weighted / cascading)
     //   - fallback strategy if primary retrieval fails

  3. QUERY AUGMENTATION EXECUTION
     ────────────────────────────
     envelope ← EMPTY_ENVELOPE(q_raw)

     FOR EACH augmentation IN plan.augmentation:
       SWITCH augmentation:
         CASE REWRITE:
           envelope ← SOTA_QUERY_REWRITE(q_raw, history, schema, ...)
           // (Algorithm from Section 3.3)

         CASE EXPAND:
           expanded, rrf ← CONTROLLED_QUERY_EXPANSION(
             envelope.generation_query.rewritten_query, ...)
           // (Algorithm from Section 4.3)
           envelope.retrieval_queries.extend(expanded)

         CASE DECOMPOSE:
           dag, schedule ← DAG_QUERY_DECOMPOSITION(
             envelope.generation_query.rewritten_query, ...)
           // (Algorithm from Section 5.3)
           envelope.generation_query.decomposed_subqueries ← dag.sub_queries()

         CASE FILTER_INJECT:
           filters ← EXTRACT_TEMPORAL_FILTERS(q_raw, analysis)
           envelope.metadata_filters.extend(filters)

  4. COLLECTION ROUTING
     ──────────────────
     routed_plan ← ROUTE_TO_COLLECTIONS(envelope, collection_registry)
     //
     // For each retrieval query in the envelope, determine:
     //   - Which collection(s) to target
     //   - What search type to use (based on collection capabilities)
     //   - What top_k to request (based on collection size and relevance estimate)
     //
     // Routing function:
     //   FOR EACH rq IN envelope.retrieval_queries:
     //     scores ← []
     //     FOR EACH coll IN collection_registry:
     //       field_overlap ← |fields(rq) ∩ fields(coll)| / |fields(rq)|
     //       schema_match  ← SCHEMA_SIM(rq.type, coll.capabilities)
     //       historical_hit← HISTORICAL_HIT_RATE(rq.intent, coll.id)
     //       score ← 0.4 * field_overlap + 0.3 * schema_match + 0.3 * historical_hit
     //       scores.append((coll, score))
     //
     //     rq.target_collections ← TOP_N(scores, n=3, threshold=0.3)

  5. RETRIEVAL EXECUTION LOOP
     ────────────────────────
     WHILE iteration < plan.max_iterations AND remaining_budget_ms > 0:

       5a. EXECUTE QUERIES (parallel)
           ────────────────────────
           results ← PARALLEL_EXECUTE(routed_plan, deadline=remaining_budget_ms * 0.6)
           //
           // For each (query, collection) pair:
           //   Use MCP tool invocation to call the appropriate search tool:
           //
           //   result = mcp.call_tool(
           //     server=collection.mcp_server,
           //     tool="search",
           //     arguments={
           //       "query": rq.query_text,
           //       "type": rq.type,
           //       "filters": envelope.metadata_filters,
           //       "limit": rq.top_k,
           //       "include_vectors": false,
           //       "include_metadata": true
           //     },
           //     timeout=per_query_timeout
           //   )
           //
           // Parallel execution with deadline propagation:
           //   All queries share a common deadline.
           //   Partial results are accepted if deadline expires.
           //   Failed queries emit structured error + continue.

       5b. RESULT FUSION
           ──────────────
           fused_results ← FUSE_RESULTS(results, rrf_config)
           //
           // Apply Reciprocal Rank Fusion across all result sets:
           //   RRF_score(d) = Σ_i [ w_i / (k + rank_i(d)) ]
           //
           // Then deduplicate by document_id,
           // apply freshness boost: score *= freshness_decay(d.timestamp),
           // apply authority boost: score *= authority_weight(d.source),
           // re-sort by final fused score,
           // truncate to top_k_final.

       5c. SUFFICIENCY EVALUATION
           ───────────────────────
           sufficiency ← EVALUATE_SUFFICIENCY(fused_results, envelope, q_raw)
           //
           // Sufficiency scoring function:
           //
           //   S(results, q) = w_coverage · COVERAGE(results, q)
           //                 + w_quality  · QUALITY(results)
           //                 + w_count    · min(|results| / min_results, 1.0)
           //
           //   COVERAGE(results, q) = fraction of sub-query facets
           //                          with ≥1 relevant result
           //                          (measured by NLI entailment score ≥ 0.7)
           //
           //   QUALITY(results)     = mean(relevance_score(d) for d in results)
           //
           //   w_coverage=0.5, w_quality=0.3, w_count=0.2
           //
           // IF S ≥ config.sufficiency_threshold (e.g., 0.75):
           //   EXIT loop → proceed to response generation
           //
           // ELSE:
           //   Identify the DEFICIT:
           //   - Which facets have insufficient coverage?
           //   - Which collections returned no results?
           //   - What filter might be too restrictive?

           IF sufficiency.score ≥ config.sufficiency_threshold:
             BREAK  // Sufficient results obtained

       5d. RE-QUERY ADAPTATION
           ────────────────────
           adaptation ← PLAN_REQUERY(sufficiency.deficit, envelope, results_cache)
           //
           // Adaptation strategies (selected based on deficit type):
           //
           //   DEFICIT: no_results_for_facet_X
           //   → Relax filters for facet X, expand query for facet X,
           //     try alternative collection
           //
           //   DEFICIT: low_relevance_scores
           //   → Rewrite query with different terminology,
           //     use HyDE for that specific facet
           //
           //   DEFICIT: overly_restrictive_filters
           //   → Remove lowest-confidence filter, widen date range
           //
           //   DEFICIT: wrong_collection
           //   → Route to next-best collection in ranking
           //
           // Apply adaptation to envelope and routed_plan:
           envelope ← APPLY_ADAPTATION(envelope, adaptation)
           routed_plan ← RE_ROUTE(envelope, collection_registry)

           action_trace.append(TraceEntry("RE_QUERY", adaptation, elapsed_ms()))
           iteration += 1
           remaining_budget_ms -= elapsed_since_last_iteration()

  6. RESPONSE SYNTHESIS
     ──────────────────
     IF plan.requires_generation:
       //
       // Assemble the synthesis prompt:
       //
       //   synthesis_context = COMPILE_PREFILL(
       //     role_policy    = "You are a precise technical assistant...",
       //     task_objective = envelope.generation_query.synthesized_intent,
       //     evidence       = FORMAT_EVIDENCE(fused_results, max_tokens=T_evidence),
       //     constraints    = [
       //       "Answer ONLY based on the provided evidence.",
       //       "Cite evidence passages by [source_id].",
       //       "If evidence is insufficient, state what is missing."
       //     ],
       //     format_spec    = dag.synthesis_strategy
       //   )
       //
       //   Token budget allocation:
       //     T_total    = context_window - T_reserved_output
       //     T_system   = min(500, 0.1 * T_total)
       //     T_evidence = min(0.7 * T_total, |fused_results_tokens|)
       //     T_history  = T_total - T_system - T_evidence

       response_text ← LLM(synthesis_context, max_tokens=T_output)
     ELSE:
       // Return structured results directly (e.g., for aggregation queries)
       response_text ← FORMAT_STRUCTURED(fused_results)

  7. PROVENANCE AND TRACE ASSEMBLY
     ─────────────────────────────
     response ← AgentResponse(
       answer         = response_text,
       sources        = fused_results.provenance_list(),
       query_trace    = action_trace,
       envelope       = envelope,
       iterations     = iteration,
       sufficiency    = sufficiency,
       total_latency  = elapsed_total_ms(),
       token_usage    = accumulate_tokens(action_trace),
       confidence     = sufficiency.score * analysis.confidence
     )

  RETURN response
```

### 6.4 Contextual Awareness: Memory-Augmented Query Resolution

The query agent maintains contextual awareness through a **tiered memory protocol**:

| Memory Layer | Purpose | Write Policy | Read Latency |
|---|---|---|---|
| **Working Memory** | Current query state, intermediate results, active sub-queries | Ephemeral, auto-expire on request completion | < 1ms (in-process) |
| **Session Memory** | Conversation history, resolved co-references, established filters | Validated per-turn, expires on session close | < 5ms (session store) |
| **Episodic Memory** | Past query patterns that led to successful retrievals for this user | Written after positive feedback, deduplicated | < 50ms (user store) |
| **Semantic Memory** | Organizational terminology mappings, query-to-collection routing priors | Admin-curated, version-controlled, validated | < 20ms (shared store) |

The agent reads from all layers during the ANALYZE phase:

$$
\text{context}_{\text{agent}} = \text{Working} \oplus \text{Session}_{[t-K:t]} \oplus \text{Episodic}_{\text{top-n}} \oplus \text{Semantic}_{\text{relevant}}
$$

Memory writes are governed by strict promotion policies:

$$
\text{Promote}(m) \iff \text{novelty}(m) > \theta_n \;\wedge\; \text{correctness\_signal}(m) = \text{TRUE} \;\wedge\; \neg\text{duplicate}(m, \mathcal{M}_{\text{existing}})
$$

---

## 7. Unified Quality Gates and Evaluation Infrastructure

### 7.1 Query Augmentation Evaluation Metrics

Every query augmentation component must be evaluated against measurable quality gates:

| Metric | Definition | Target | Measurement |
|---|---|---|---|
| **Intent Preservation Index (IPI)** | $\frac{\|E(q) \cap E(q')\|}{\|E(q)\|} \cdot \cos(\mathbf{v}(q), \mathbf{v}(q'))$ | $\geq 0.85$ | Computed per-rewrite |
| **Retrieval Recall Lift** | $\frac{\text{Recall}@k(q') - \text{Recall}@k(q)}{\text{Recall}@k(q)}$ | $\geq +15\%$ | Measured on eval set |
| **Expansion Diversity** | $1 - \frac{1}{\binom{n}{2}}\sum_{i<j} \cos(\mathbf{v}(q_i'), \mathbf{v}(q_j'))$ | $\geq 0.3$ | Per expansion set |
| **Decomposition Coverage** | $\frac{\|\text{facets\_covered}\|}{\|\text{facets\_required}\|}$ | $= 1.0$ | Per decomposition |
| **Agent Loop Efficiency** | $\frac{\text{sufficiency\_score}}{\text{iterations} \cdot \text{cost}}$ | Maximized | Per request |
| **End-to-End Latency** | Total wall-clock from $q_{\text{raw}}$ to $\text{response}$ | $\leq \tau_{\text{SLA}}$ | p50, p95, p99 |
| **Drift Violation Rate** | $\frac{\|\{q' \mid \text{IPI}(q, q') < \theta\}\|}{\|\text{total\_augmentations}\|}$ | $\leq 2\%$ | Continuous monitoring |

### 7.2 CI/CD Integration

```
PIPELINE: QUERY_AUGMENTATION_EVAL_CI
────────────────────────────────────

Trigger: on_commit to query_augmentation/* OR weekly_schedule

Stages:

  1. REPLAY_SET_EXECUTION
     ─────────────────────
     FOR EACH (q_raw, expected_results, ground_truth_intent) IN eval_corpus:
       envelope ← QUERY_AGENT_ORCHESTRATOR(q_raw, ...)
       metrics  ← COMPUTE_METRICS(envelope, expected_results, ground_truth_intent)
       store(metrics, run_id)

  2. REGRESSION_DETECTION
     ─────────────────────
     baseline ← load_metrics(baseline_run_id)
     current  ← load_metrics(current_run_id)
     FOR EACH metric IN [IPI, recall_lift, diversity, coverage, latency]:
       delta ← current[metric] - baseline[metric]
       IF delta < -regression_threshold[metric]:
         FAIL_BUILD("Regression detected: {metric} degraded by {delta}")

  3. DRIFT_AUDIT
     ────────────
     drift_violations ← current.filter(IPI < θ_IPI)
     IF |drift_violations| / |eval_corpus| > 0.02:
       FAIL_BUILD("Drift violation rate exceeds 2%")

  4. COST_BUDGET_CHECK
     ──────────────────
     avg_tokens ← mean(current.token_usage)
     avg_calls  ← mean(current.llm_calls)
     IF avg_tokens > token_budget_per_query OR avg_calls > max_calls_per_query:
       WARN("Cost budget exceeded: {avg_tokens} tokens, {avg_calls} calls")

  5. ARTIFACT_PUBLICATION
     ─────────────────────
     publish_metrics(current, dashboard)
     publish_replay_traces(current, trace_store)
     update_baseline_if_passing(current)
```

---

## 8. Production Reliability Engineering

### 8.1 Failure Modes and Mitigations

| Failure Mode | Detection | Mitigation | Recovery |
|---|---|---|---|
| **LLM rewrite timeout** | Deadline exceeded | Fall back to co-reference-resolved $q_{\text{resolved}}$ | Continue with degraded quality; log event |
| **LLM generates hallucinated filters** | Filter predicate references non-existent schema field | Validate all filters against schema before execution | Drop invalid filters; emit structured warning |
| **Expansion drift** | IPI below threshold | Reject drifted expansions; use only validated subset | Minimum viable expansion set (original + 1 HyDE) |
| **Decomposition cycle** | Cycle detected in DAG | Break back-edges by confidence; re-validate DAG | Flatten to parallel independent sub-queries |
| **Agent loop divergence** | Iterations exceed $n_{\text{max}}$ with no sufficiency improvement | Force-terminate loop; return best available results | Return partial results with explicit insufficiency flag |
| **All collections return empty** | Zero results across all routed collections | Widen filters, remove metadata constraints, try broader collection | If still empty, return "no information found" with suggested reformulations |
| **Token budget exhaustion** | Accumulated tokens exceed budget | Terminate current expansion/decomposition step | Proceed with results obtained so far |

### 8.2 Idempotency and Determinism

Query augmentation pipelines must be **idempotent**: applying the same augmentation twice must produce equivalent results. This is enforced by:

1. **Temperature control**: All LLM calls in the augmentation pipeline use $T \leq 0.3$ for near-deterministic outputs.
2. **Seed pinning**: Where supported, fix the random seed per request ID for reproducibility.
3. **Cache-first execution**: Hash the (query, history, schema) tuple; return cached augmentation result if available within TTL.

$$
\text{cache\_key} = \text{SHA256}\left(q_{\text{raw}} \| \text{hash}(\mathcal{H}_{[-K:]}) \| \text{schema\_version}\right)
$$

### 8.3 Observability Contract

Every query augmentation invocation emits a structured trace conforming to:

```json
{
  "trace_id": "uuid",
  "span_name": "query_augmentation",
  "spans": [
    {"name": "coreference_resolution", "duration_ms": 45, "tokens_in": 320, "tokens_out": 80},
    {"name": "intent_classification",  "duration_ms": 120, "tokens_in": 450, "tokens_out": 50, "result": "DIAGNOSTIC"},
    {"name": "step_back_abstraction",  "duration_ms": 180, "tokens_in": 500, "tokens_out": 90},
    {"name": "entity_extraction",      "duration_ms": 95,  "tokens_in": 400, "tokens_out": 120},
    {"name": "keyword_densification",  "duration_ms": 150, "tokens_in": 380, "tokens_out": 60, "density_score": 0.42},
    {"name": "expansion_hyde",         "duration_ms": 200, "tokens_in": 450, "tokens_out": 150},
    {"name": "expansion_perspectives", "duration_ms": 350, "tokens_in": 900, "tokens_out": 240, "accepted": 3, "rejected": 1},
    {"name": "drift_validation",       "duration_ms": 15,  "ipi_score": 0.91, "passed": true},
    {"name": "collection_routing",     "duration_ms": 8,   "collections": ["docs_v2", "kb_internal"]}
  ],
  "total_duration_ms": 1163,
  "total_tokens": 3690,
  "augmentation_methods": ["REWRITE", "EXPAND_HYDE", "EXPAND_PERSPECTIVE"],
  "drift_score": 0.09,
  "final_retrieval_queries": 5
}
```

---

## 9. Architectural Integration: Where Query Augmentation Sits in the Agentic Stack

```
┌──────────────────────────────────────────────────────────────────┐
│                        USER / APPLICATION                        │
│                     (JSON-RPC boundary)                           │
└────────────────────────────┬─────────────────────────────────────┘
                             │ q_raw + session_id
                             ▼
┌──────────────────────────────────────────────────────────────────┐
│                   QUERY AUGMENTATION LAYER                        │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────────────┐ │
│  │ Co-Ref   │→ │ Intent   │→ │ Rewrite  │→ │ Expand /         │ │
│  │ Resolve  │  │ Classify │  │ + Dense  │  │ Decompose /      │ │
│  │          │  │          │  │          │  │ Agent Orchestrate │ │
│  └──────────┘  └──────────┘  └──────────┘  └────────┬─────────┘ │
│                                                      │           │
│  Output: AugmentedQueryEnvelope (typed Protobuf)     │           │
└──────────────────────────────────────────────────────┼───────────┘
                                                       │
                             ┌─────────────────────────▼──────────┐
                             │      RETRIEVAL ENGINE               │
                             │  (hybrid: semantic + keyword +      │
                             │   graph + metadata filters)         │
                             │  Collection routing via MCP         │
                             │  RRF fusion + freshness/authority   │
                             └─────────────────────────┬──────────┘
                                                       │
                             ┌─────────────────────────▼──────────┐
                             │      CONTEXT COMPILER (PREFILL)     │
                             │  Assemble: role + evidence +        │
                             │  memory + constraints + format      │
                             │  Budget: explicit token allocation  │
                             └─────────────────────────┬──────────┘
                                                       │
                             ┌─────────────────────────▼──────────┐
                             │      GENERATION + VERIFICATION      │
                             │  LLM synthesis → critique →         │
                             │  repair → commit                    │
                             └─────────────────────────┬──────────┘
                                                       │
                             ┌─────────────────────────▼──────────┐
                             │      RESPONSE DELIVERY              │
                             │  + provenance + traces + feedback   │
                             └────────────────────────────────────┘
```

---

## 10. Summary of Key Invariants

| Invariant | Enforcement Mechanism |
|---|---|
| No raw query reaches the retrieval engine | Pipeline architecture: augmentation layer is mandatory |
| Every augmented query has provenance | `QueryProvenance` field is required in Protobuf schema |
| Drift never exceeds $\delta_{\text{max}}$ | IPI validation gate with automatic fallback |
| Expansion cardinality is bounded | MMR selection with hard cap $n_{\text{max}}$ |
| Decomposition depth is bounded | DAG depth enforcement with flatten-to-depth |
| Agent loop terminates | Iteration cap + deadline + token budget |
| All failures are observable | Structured trace emission on every span |
| Augmentation is idempotent | Cache-first with content-addressed keys |
| Cost is bounded per request | Token and LLM-call budget enforcement |
| Quality is continuously validated | CI pipeline with regression detection on eval corpus |

---

This report establishes query augmentation as a **typed, bounded, observable subsystem** within the agentic context engineering stack—not an ad hoc preprocessing step. Every transformation is governed by formal quality constraints, drift bounds, latency budgets, and typed protocol contracts, ensuring that the system operates predictably, safely, and cost-efficiently at production scale. The mathematical formulations, pseudo-algorithms, and architectural contracts provided herein constitute a complete implementation specification for a principal-level engineering team.



# Query Augmentation as a Deterministic Context Engineering Subsystem: A Principal-Level Technical Report

---

## Abstract

Query augmentation constitutes the first critical control surface in any production agentic retrieval pipeline. Raw user queries exhibit distributional drift from the latent information need, introducing systematic retrieval degradation that propagates irreversibly through downstream stages. This report formalizes query rewriting, query expansion, query decomposition, and query agent orchestration as typed, bounded transformations within a compiled context engineering stack. Each technique is specified with mathematical formulations, pseudocode algorithms, failure mode analysis, and production-grade architectural constraints. The objective is to establish query augmentation not as a heuristic preprocessing step, but as a **deterministic query compilation layer** with provenance, measurability, and mechanical correctness guarantees.

---

## 1. Problem Formalization: The Query-Intent Divergence Gap

### 1.1 Formal Definition

Let $q_{\text{raw}}$ denote the raw user query string, and let $\mathcal{I}$ denote the latent information need (unobservable). Define the **query-intent divergence** as:

$$
\Delta(q_{\text{raw}}, \mathcal{I}) = 1 - \frac{|\text{Rel}(q_{\text{raw}}) \cap \text{Rel}(\mathcal{I})|}{|\text{Rel}(\mathcal{I})|}
$$

where $\text{Rel}(x)$ denotes the set of documents relevant under query or intent $x$. When $\Delta \to 1$, the raw query retrieves almost none of the documents the user actually needs. The purpose of query augmentation is to produce a transformed query $q^{*}$ such that:

$$
\Delta(q^{*}, \mathcal{I}) \ll \Delta(q_{\text{raw}}, \mathcal{I})
$$

### 1.2 The Irreversible Degradation Theorem

**Claim:** In a pipeline $\mathcal{P} = \text{QueryAug} \to \text{Retrieve} \to \text{Rerank} \to \text{Synthesize}$, retrieval quality is bounded above by the fidelity of the query augmentation stage.

Let $\text{Recall}@k$ denote recall at retrieval depth $k$. For any downstream reranker $\mathcal{R}$:

$$
\text{Precision}_{\mathcal{R}}(q) \leq \frac{|\text{Rel}(\mathcal{I}) \cap \text{Retrieved}_k(q)|}{k}
$$

If $\text{Retrieved}_k(q_{\text{raw}}) \cap \text{Rel}(\mathcal{I}) = \emptyset$, no reranker or prompt engineering recovers relevance. This is the formal statement of the **"garbage in, garbage out" invariant**: query augmentation failure is non-recoverable downstream.

### 1.3 Taxonomy of Raw Query Pathologies

| Pathology Class | Formal Characterization | Example |
|---|---|---|
| **Lexical Ambiguity** | $|S(q_{\text{raw}})| > 1$ where $S$ maps to semantic interpretations | "How do I make this work?" |
| **Referential Incompleteness** | Unresolved anaphora, missing entity binding | "Fix the error from before" |
| **Schema Mismatch** | $q_{\text{raw}} \notin \mathcal{L}_{\text{retrieval}}$ where $\mathcal{L}$ is the retrieval system's effective query language | Conversational prose vs. keyword index |
| **Multi-Intent Entanglement** | $\mathcal{I} = \{\mathcal{I}_1, \mathcal{I}_2, \ldots, \mathcal{I}_n\}$, $n > 1$ | "Compare X and Y, then show me how to deploy Z" |
| **Noise Contamination** | $\text{SNR}(q_{\text{raw}}) < \tau_{\min}$ | Filler words, emotional language, typos |

---

## 2. Query Rewriting: Deterministic Intent Normalization

### 2.1 Formal Specification

Query rewriting is a function $\mathcal{W}: \Sigma^{*} \times \mathcal{C} \to \Sigma^{*}$ where $\Sigma^{*}$ is the space of all query strings and $\mathcal{C}$ is the conversational/session context. The rewriter must satisfy three invariants:

**Invariant 1 — Semantic Preservation:**
$$
\text{sim}_{\text{semantic}}(q_{\text{raw}}, \mathcal{W}(q_{\text{raw}}, \mathcal{C})) \geq \theta_{\text{preserve}}
$$

**Invariant 2 — Retrieval Utility Gain:**
$$
\text{Recall}@k(\mathcal{W}(q_{\text{raw}}, \mathcal{C})) > \text{Recall}@k(q_{\text{raw}})
$$

**Invariant 3 — Noise Reduction:**
$$
H(\mathcal{W}(q_{\text{raw}}, \mathcal{C})) < H(q_{\text{raw}})
$$

where $H$ denotes entropy over token distribution relative to the retrieval index vocabulary.

### 2.2 The Rewrite-Retrieve-Read Paradigm

The classical Retrieve-then-Read pipeline is superseded by a three-stage compiled pipeline:

$$
q_{\text{raw}} \xrightarrow{\mathcal{W}} q_{\text{rewritten}} \xrightarrow{\mathcal{R}} \mathcal{D}_k \xrightarrow{\mathcal{G}} \text{Response}
$$

where $\mathcal{R}$ is the retrieval function returning top-$k$ documents $\mathcal{D}_k$, and $\mathcal{G}$ is the generative synthesis function.

### 2.3 Rewriting Operations Taxonomy

Define the rewriter as a composition of atomic operations:

$$
\mathcal{W} = \phi_{\text{resolve}} \circ \phi_{\text{denoise}} \circ \phi_{\text{enrich}} \circ \phi_{\text{restructure}}
$$

| Operation | Symbol | Formal Action |
|---|---|---|
| **Restructuring** | $\phi_{\text{restructure}}$ | Parse syntactic tree, normalize to declarative/interrogative canonical form |
| **Denoising** | $\phi_{\text{denoise}}$ | Remove tokens $t_i$ where $\text{TF-IDF}(t_i, \mathcal{D}_{\text{corpus}}) < \epsilon$ or $\text{PMI}(t_i, \mathcal{I}) < \delta$ |
| **Keyword Enrichment** | $\phi_{\text{enrich}}$ | Inject terms $t_j$ from domain ontology $\mathcal{O}$ where $\text{sim}(t_j, q_{\text{raw}}) > \gamma$ |
| **Coreference Resolution** | $\phi_{\text{resolve}}$ | Bind anaphoric references using session context $\mathcal{C}$ |

### 2.4 Pseudocode: Production Query Rewriter

```
ALGORITHM: DeterministicQueryRewriter
────────────────────────────────────────────────────────────────
Input:
    q_raw       : str                    // Raw user query
    C_session   : SessionContext         // Conversation history, entity bindings
    O_domain    : DomainOntology         // Typed domain vocabulary graph
    θ_preserve  : float                  // Semantic preservation threshold
    budget_ms   : int                    // Latency budget in milliseconds

Output:
    q_rewritten : RewrittenQuery         // Typed output with provenance

Procedure:
    1. COREFERENCE RESOLUTION
       entities ← ExtractEntities(q_raw)
       unresolved ← FilterUnresolved(entities)
       FOR EACH ref IN unresolved:
           binding ← ResolveFromSessionContext(ref, C_session)
           IF binding IS NULL:
               binding ← ResolveFromOntology(ref, O_domain)
           IF binding IS NULL:
               FLAG ref AS AMBIGUOUS  // Propagate uncertainty
           q_raw ← Substitute(q_raw, ref, binding)

    2. NOISE REDUCTION
       tokens ← Tokenize(q_raw)
       signal_tokens ← []
       FOR EACH t IN tokens:
           IF InformationGain(t, O_domain) ≥ ε_min:
               signal_tokens.APPEND(t)
           ELSE:
               LOG(provenance="denoised", token=t, reason="low IG")
       q_denoised ← Reconstruct(signal_tokens)

    3. KEYWORD ENRICHMENT
       candidate_terms ← OntologyExpand(q_denoised, O_domain, max_hops=2)
       enriched_terms ← []
       FOR EACH t_j IN candidate_terms:
           IF CosineSimilarity(Embed(t_j), Embed(q_denoised)) ≥ γ:
               enriched_terms.APPEND(t_j)
           IF |enriched_terms| ≥ MAX_ENRICHMENT_TERMS:
               BREAK
       q_enriched ← Concatenate(q_denoised, enriched_terms)

    4. STRUCTURAL NORMALIZATION
       q_rewritten ← LLMRewrite(
           prompt = COMPILED_REWRITE_PROMPT,
           input  = q_enriched,
           constraints = {
               "preserve_intent": True,
               "output_format": "keyword-dense declarative",
               "max_tokens": REWRITE_TOKEN_BUDGET
           }
       )

    5. SEMANTIC PRESERVATION VERIFICATION
       sim ← CosineSimilarity(Embed(q_raw), Embed(q_rewritten))
       IF sim < θ_preserve:
           LOG(severity="WARN", event="rewrite_drift", sim=sim)
           q_rewritten ← q_raw  // Fallback to original on drift

    6. PROVENANCE ANNOTATION
       q_rewritten.provenance ← {
           "original": q_raw,
           "operations": [resolve, denoise, enrich, restructure],
           "semantic_sim": sim,
           "latency_ms": ElapsedTime(),
           "model_version": REWRITER_MODEL_ID
       }

    RETURN q_rewritten
────────────────────────────────────────────────────────────────
```

### 2.5 Semantic Preservation Verification Gate

The verification gate is non-negotiable. Without it, the rewriter may introduce **semantic drift** — a condition where the rewritten query diverges from user intent. The gate operates as:

$$
\text{PASS}(q^{*}) \iff \text{sim}(\mathbf{e}(q_{\text{raw}}), \mathbf{e}(q^{*})) \geq \theta_{\text{preserve}} \land \text{EntityCoverage}(q^{*}, q_{\text{raw}}) = 1.0
$$

where $\mathbf{e}(\cdot)$ is the embedding function (e.g., a contrastive encoder trained on query-document pairs, not a general-purpose sentence encoder). $\text{EntityCoverage}$ verifies that all named entities and critical noun phrases from $q_{\text{raw}}$ survive the rewrite.

---

## 3. Query Expansion: Controlled Multi-Hypothesis Retrieval

### 3.1 Formal Definition

Query expansion maps a single query to a set of $n$ queries:

$$
\mathcal{E}: q \to \{q_1, q_2, \ldots, q_n\}
$$

such that the union of retrieved documents increases recall:

$$
\text{Recall}\left(\bigcup_{i=1}^{n} \text{Ret}_k(q_i)\right) \geq \text{Recall}(\text{Ret}_k(q))
$$

### 3.2 Expansion Strategies

#### 3.2.1 Generative Expansion (LLM-Based)

An LLM generates paraphrases and related formulations:

$$
\{q_1, \ldots, q_n\} = \text{LLM}(q, \text{prompt}_{\text{expand}}, T)
$$

where $T$ is the temperature parameter controlling lexical diversity. Higher $T$ yields broader lexical coverage but increases **query drift risk**.

#### 3.2.2 Hypothetical Document Embedding (HyDE)

Generate a hypothetical answer $a_{\text{hyp}}$ and use its embedding for retrieval:

$$
a_{\text{hyp}} = \text{LLM}(q, \text{prompt}_{\text{hyde}})
$$
$$
\mathcal{D}_k = \text{Ret}_k(\mathbf{e}(a_{\text{hyp}}))
$$

**Trade-off Analysis:** HyDE shifts the retrieval target from query-space to answer-space, which can dramatically improve recall for complex questions but introduces hallucinated content into the retrieval embedding. The mitigation is to use $a_{\text{hyp}}$ exclusively as an embedding anchor, never as a source of factual content.

#### 3.2.3 Ontology-Grounded Expansion

$$
\{q_1, \ldots, q_n\} = \{q \oplus t \mid t \in \mathcal{N}_k(q, \mathcal{O})\}
$$

where $\mathcal{N}_k(q, \mathcal{O})$ returns the $k$-nearest terms in domain ontology $\mathcal{O}$ via graph traversal (synonyms, hypernyms, meronyms).

### 3.3 Formal Risk Model for Query Expansion

Define three risk metrics that must be bounded:

**Query Drift:**
$$
\text{Drift}(q_i, q) = 1 - \text{sim}(\mathbf{e}(q_i), \mathbf{e}(q))
$$

Constraint: $\forall i: \text{Drift}(q_i, q) \leq \delta_{\text{max}}$

**Over-Expansion Penalty:**
$$
\text{Precision}@k \propto \frac{1}{n} \quad \text{as } n \to \infty
$$

There exists an optimal $n^{*}$ that maximizes $F_1$:

$$
n^{*} = \arg\max_n \left[ \frac{2 \cdot \text{Precision}@k(n) \cdot \text{Recall}@k(n)}{\text{Precision}@k(n) + \text{Recall}@k(n)} \right]
$$

**Computational Overhead:**
$$
\text{Latency}(\mathcal{E}) = \text{Latency}_{\text{generate}}(n) + \sum_{i=1}^{n} \text{Latency}_{\text{retrieve}}(q_i)
$$

Mitigation: Parallelize retrieval across expanded queries with a fork-join executor bounded by $\text{Latency}_{\max}$.

### 3.4 Pseudocode: Bounded Query Expansion with Drift Control

```
ALGORITHM: BoundedQueryExpansion
────────────────────────────────────────────────────────────────
Input:
    q           : RewrittenQuery
    n_max       : int                    // Maximum expansion count
    δ_max       : float                  // Maximum drift tolerance
    latency_ms  : int                    // Total latency budget
    strategy    : Enum{GENERATIVE, HYDE, ONTOLOGY, HYBRID}

Output:
    Q_expanded  : List[ExpandedQuery]    // Each with drift score & provenance

Procedure:
    1. GENERATE CANDIDATES
       SWITCH strategy:
           CASE GENERATIVE:
               candidates ← LLMExpand(q, n=n_max*2, T=0.7)
           CASE HYDE:
               a_hyp ← LLMGenerateHypotheticalAnswer(q)
               candidates ← [q, a_hyp] + LLMExpand(q, n=n_max-2, T=0.5)
           CASE ONTOLOGY:
               terms ← OntologyTraverse(q, max_hops=2, max_terms=n_max*3)
               candidates ← [q ⊕ t FOR t IN terms]
           CASE HYBRID:
               candidates ← UNION(GENERATIVE, ONTOLOGY)

    2. DRIFT FILTERING
       Q_filtered ← []
       FOR EACH q_i IN candidates:
           drift_i ← 1 - CosineSimilarity(Embed(q_i), Embed(q))
           IF drift_i ≤ δ_max:
               q_i.drift_score ← drift_i
               Q_filtered.APPEND(q_i)

    3. DIVERSITY-AWARE SELECTION (MMR)
       Q_expanded ← MaximalMarginalRelevance(
           candidates = Q_filtered,
           query_embedding = Embed(q),
           λ = 0.6,          // Balance relevance vs. diversity
           k = n_max
       )

    4. DEDUPLICATION
       Q_expanded ← DeduplicateBySemantic(Q_expanded, sim_threshold=0.95)

    5. PROVENANCE & TOKEN BUDGET
       total_tokens ← SUM(TokenCount(q_i) FOR q_i IN Q_expanded)
       IF total_tokens > TOKEN_BUDGET_EXPANSION:
           Q_expanded ← TruncateByDriftScore(Q_expanded, TOKEN_BUDGET_EXPANSION)

       FOR EACH q_i IN Q_expanded:
           q_i.provenance ← {
               "parent_query": q.id,
               "strategy": strategy,
               "drift": q_i.drift_score,
               "mmr_rank": q_i.rank
           }

    RETURN Q_expanded
────────────────────────────────────────────────────────────────
```

### 3.5 Maximal Marginal Relevance (MMR) for Expansion Selection

To prevent redundant expanded queries from consuming retrieval budget:

$$
\text{MMR}(q_i) = \lambda \cdot \text{sim}(q_i, q) - (1 - \lambda) \cdot \max_{q_j \in S} \text{sim}(q_i, q_j)
$$

where $S$ is the already-selected set, and $\lambda \in [0, 1]$ trades off relevance against diversity. This ensures the expansion set covers distinct retrieval neighborhoods in embedding space.

---

## 4. Query Decomposition: Multi-Intent Factorization

### 4.1 Formal Model

Given a complex query $q_c$ with latent intent set $\mathcal{I} = \{\mathcal{I}_1, \ldots, \mathcal{I}_m\}$, decomposition produces:

$$
\mathcal{F}: q_c \to \{s_1, s_2, \ldots, s_m\}
$$

such that:

$$
\bigcup_{j=1}^{m} \text{Rel}(s_j) \supseteq \text{Rel}(q_c) \quad \text{and} \quad \forall j: |\mathcal{I}(s_j)| = 1
$$

Each sub-query $s_j$ is **mono-intent** — targeting exactly one information need.

### 4.2 Decomposition Dependency Graph

Sub-queries may have execution dependencies. Define a **decomposition DAG** $G_D = (V, E)$ where:

- $V = \{s_1, \ldots, s_m\}$
- $(s_i, s_j) \in E \iff s_j$ requires the result of $s_i$ as input

**Topological execution order** is mandatory: sub-queries are scheduled according to $\text{TopoSort}(G_D)$, with independent sub-queries eligible for parallel execution.

### 4.3 Decomposition Quality Metrics

**Coverage:**
$$
\text{Coverage}(\mathcal{F}, q_c) = \frac{|\bigcup_j \text{Rel}(s_j) \cap \text{Rel}(q_c)|}{|\text{Rel}(q_c)|}
$$

**Orthogonality:**
$$
\text{Orthogonality}(\mathcal{F}) = 1 - \frac{1}{\binom{m}{2}} \sum_{i < j} \text{sim}(\mathbf{e}(s_i), \mathbf{e}(s_j))
$$

Higher orthogonality indicates less redundancy across sub-queries.

**Atomicity:**
$$
\text{Atomicity}(s_j) = \begin{cases} 1 & \text{if } |\mathcal{I}(s_j)| = 1 \\ 0 & \text{otherwise} \end{cases}
$$

All three metrics must exceed defined thresholds for the decomposition to be admitted into the retrieval pipeline.

### 4.4 Pseudocode: DAG-Based Query Decomposition with Synthesis

```
ALGORITHM: DAGQueryDecomposition
────────────────────────────────────────────────────────────────
Input:
    q_c             : RewrittenQuery         // Complex multi-intent query
    max_depth       : int                    // Maximum decomposition depth
    θ_orthogonality : float                  // Minimum orthogonality threshold
    θ_atomicity     : float                  // Minimum atomicity (1.0 for strict)

Output:
    result          : SynthesizedResponse
    G_D             : DecompositionDAG        // With execution trace

Procedure:
    1. INTENT ANALYSIS
       intents ← LLMIntentExtract(
           q_c,
           prompt = DECOMPOSITION_ANALYSIS_PROMPT,
           output_schema = {
               "sub_queries": List[{
                   "id": str,
                   "query": str,
                   "intent_label": str,
                   "depends_on": List[str],
                   "retrieval_target": Enum{VECTOR, KEYWORD, GRAPH, API}
               }]
           }
       )

    2. CONSTRUCT DEPENDENCY DAG
       G_D ← DirectedAcyclicGraph()
       FOR EACH sq IN intents.sub_queries:
           G_D.AddNode(sq.id, sq)
           FOR EACH dep IN sq.depends_on:
               G_D.AddEdge(dep, sq.id)
       ASSERT G_D.IsAcyclic()  // Hard invariant

    3. QUALITY GATE
       orth ← ComputeOrthogonality(intents.sub_queries)
       IF orth < θ_orthogonality:
           // Merge highly overlapping sub-queries
           intents ← MergeOverlapping(intents, sim_threshold=0.85)
           G_D ← Rebuild(intents)

       FOR EACH sq IN intents.sub_queries:
           atom ← LLMAtomicityCheck(sq.query)
           IF atom < θ_atomicity AND current_depth < max_depth:
               // Recursive decomposition
               sub_results ← DAGQueryDecomposition(
                   sq, max_depth - 1, θ_orthogonality, θ_atomicity
               )
               G_D ← Splice(G_D, sq.id, sub_results.G_D)

    4. PARALLEL EXECUTION WITH DEPENDENCY RESPECT
       execution_order ← TopologicalSort(G_D)
       results_map ← {}
       FOR EACH level IN execution_order.parallel_levels():
           PARALLEL FOR EACH sq IN level:
               // Inject dependency results into sub-query context
               dep_context ← [results_map[d] FOR d IN sq.depends_on]
               sq_augmented ← InjectDependencyContext(sq, dep_context)

               // Route to appropriate retrieval backend
               SWITCH sq.retrieval_target:
                   CASE VECTOR:
                       docs ← SemanticRetrieve(sq_augmented, k=TOP_K)
                   CASE KEYWORD:
                       docs ← BM25Retrieve(sq_augmented, k=TOP_K)
                   CASE GRAPH:
                       docs ← GraphTraverse(sq_augmented, hops=2)
                   CASE API:
                       docs ← ToolInvoke(sq_augmented)

               results_map[sq.id] ← {
                   "query": sq,
                   "documents": docs,
                   "retrieval_target": sq.retrieval_target,
                   "latency_ms": elapsed
               }

    5. CROSS-SUB-QUERY SYNTHESIS
       synthesis_context ← CompileResults(results_map, G_D)
       result ← LLMSynthesize(
           prompt = SYNTHESIS_PROMPT,
           context = synthesis_context,
           original_query = q_c,
           constraints = {
               "must_address_all_intents": True,
               "cite_sources": True,
               "max_tokens": SYNTHESIS_TOKEN_BUDGET
           }
       )

    6. COMPLETENESS VERIFICATION
       coverage ← VerifyCoverage(result, intents.sub_queries)
       IF coverage < COVERAGE_THRESHOLD:
           missing ← IdentifyMissing(result, intents.sub_queries)
           FOR EACH m IN missing:
               补充_docs ← FallbackRetrieve(m, strategy=BROADER)
               result ← IncrementalSynthesize(result, 补充_docs, m)

    RETURN result, G_D
────────────────────────────────────────────────────────────────
```

### 4.5 Synthesis Aggregation Strategy

After independent sub-query retrieval, the system must produce a unified response. Define the synthesis function:

$$
\mathcal{G}_{\text{synth}}(\{(s_j, \mathcal{D}_j)\}_{j=1}^{m}, q_c) \to r
$$

The synthesis prompt is compiled as a deterministic prefill containing:

1. **Original query** $q_c$ as the grounding anchor
2. **Sub-query results** $\{(s_j, \mathcal{D}_j)\}$ ordered by DAG execution topology
3. **Cross-reference instructions** requiring the model to reconcile contradictions between sub-query results
4. **Citation mandates** linking every factual claim to a source document with provenance ID

---

## 5. Query Agents: Closed-Loop Adaptive Query Orchestration

### 5.1 Architectural Specification

A Query Agent is a **bounded control loop** that subsumes rewriting, expansion, decomposition, and retrieval routing as tool-callable actions within an agent executor. It operates as a finite-state machine with typed transitions:

$$
\mathcal{A}_q: (q_{\text{raw}}, \mathcal{C}, \mathcal{S}) \to (r, \tau)
$$

where $\mathcal{S}$ is the full schema registry of available collections, $r$ is the final response, and $\tau$ is the execution trace.

### 5.2 State Machine Definition

```
States:
    ANALYZE     → Intent extraction and schema matching
    ROUTE       → Collection selection and retrieval strategy determination
    CONSTRUCT   → Dynamic query construction with filters, aggregations
    EXECUTE     → Query execution against selected backends
    EVALUATE    → Result sufficiency assessment
    REPAIR      → Re-query with adjusted parameters
    SYNTHESIZE  → Response generation
    COMMIT      → Final output with trace

Transitions:
    ANALYZE     → ROUTE       [always]
    ROUTE       → CONSTRUCT   [collection(s) selected]
    CONSTRUCT   → EXECUTE     [query compiled]
    EXECUTE     → EVALUATE    [results returned]
    EVALUATE    → SYNTHESIZE  [sufficiency ≥ θ_sufficiency]
    EVALUATE    → REPAIR      [sufficiency < θ_sufficiency]
    REPAIR      → ROUTE       [try alternative collection]
    REPAIR      → CONSTRUCT   [adjust query parameters]
    REPAIR      → ANALYZE     [re-interpret intent, max 1 re-analysis]
    SYNTHESIZE  → COMMIT      [always]

Invariants:
    max_loops(EVALUATE → REPAIR → *) ≤ MAX_REPAIR_ITERATIONS
    total_latency ≤ LATENCY_BUDGET_MS
    total_tokens_consumed ≤ TOKEN_BUDGET_AGENT
```

### 5.3 Formal Decision Functions

#### 5.3.1 Collection Routing

Given collections $\mathcal{K} = \{K_1, \ldots, K_p\}$ with schema descriptors $\{\sigma_1, \ldots, \sigma_p\}$, the routing function selects:

$$
K^{*} = \arg\max_{K_i} \left[ \alpha \cdot \text{SchemaRelevance}(\sigma_i, q) + \beta \cdot \text{DataFreshness}(K_i) + \gamma \cdot \text{HistoricalYield}(K_i, q_{\text{class}}) \right]
$$

where $\alpha + \beta + \gamma = 1$ and $\text{HistoricalYield}$ is computed from episodic memory of past query-collection success rates.

#### 5.3.2 Sufficiency Evaluation

After retrieval, the agent evaluates whether the results $\mathcal{D}_k$ are sufficient to answer $q$:

$$
\text{Sufficiency}(\mathcal{D}_k, q) = \frac{1}{|\mathcal{I}|} \sum_{j=1}^{|\mathcal{I}|} \mathbb{1}\left[\exists d \in \mathcal{D}_k : \text{Covers}(d, \mathcal{I}_j)\right]
$$

If $\text{Sufficiency} < \theta_{\text{suff}}$, the agent transitions to REPAIR.

#### 5.3.3 Dynamic Filter Construction

The agent constructs metadata filters dynamically based on extracted entities and schema knowledge:

$$
\mathcal{F}_{\text{dynamic}} = \bigwedge_{i} \left( \text{field}_i \; \text{op}_i \; \text{value}_i \right)
$$

where each predicate is derived from entity extraction on the query and schema-aware type matching.

### 5.4 Pseudocode: Full Query Agent Executor

```
ALGORITHM: QueryAgentExecutor
────────────────────────────────────────────────────────────────
Input:
    q_raw           : str
    C_session       : SessionContext
    schema_registry : SchemaRegistry        // All collection schemas
    tools           : ToolRegistry          // MCP-discovered tools
    config          : AgentConfig {
        max_repair_iterations : int = 3,
        θ_sufficiency         : float = 0.8,
        latency_budget_ms     : int = 5000,
        token_budget          : int = 8000
    }

Output:
    response        : TypedResponse
    trace           : ExecutionTrace

Procedure:
    state ← ANALYZE
    trace ← ExecutionTrace()
    repair_count ← 0
    token_consumed ← 0
    deadline ← Now() + config.latency_budget_ms

    WHILE state ≠ COMMIT AND Now() < deadline:
        SWITCH state:

            CASE ANALYZE:
                // Intent extraction with schema awareness
                analysis ← LLMAnalyze(
                    query = q_raw,
                    session = C_session,
                    available_schemas = schema_registry.Summaries(),
                    output_schema = AnalysisSchema {
                        intents: List[Intent],
                        entities: List[Entity],
                        requires_aggregation: bool,
                        requires_search: bool,
                        suggested_collections: List[str],
                        complexity: Enum{SIMPLE, COMPOUND, MULTI_HOP}
                    }
                )
                token_consumed += analysis.tokens_used
                trace.Log(ANALYZE, analysis)

                // If compound, decompose first
                IF analysis.complexity == MULTI_HOP:
                    sub_queries ← DAGQueryDecomposition(q_raw)
                    // Execute each sub-query through agent recursively
                    // with depth bound
                    results ← []
                    FOR EACH sq IN sub_queries:
                        sub_result ← QueryAgentExecutor(
                            sq, C_session, schema_registry, tools,
                            config.WithReducedBudget()
                        )
                        results.APPEND(sub_result)
                    state ← SYNTHESIZE
                    CONTINUE

                state ← ROUTE

            CASE ROUTE:
                // Score each collection
                scored_collections ← []
                FOR EACH K_i IN schema_registry.Collections():
                    score ← (
                        α * SchemaRelevance(K_i.schema, analysis) +
                        β * DataFreshness(K_i.metadata) +
                        γ * EpisodicYield(K_i.id, analysis.intents)
                    )
                    scored_collections.APPEND((K_i, score))

                selected ← TopK(scored_collections, k=MAX_COLLECTIONS)
                trace.Log(ROUTE, selected)
                state ← CONSTRUCT

            CASE CONSTRUCT:
                queries ← []
                FOR EACH (K_i, _) IN selected:
                    // Build typed query per collection
                    typed_query ← DynamicQueryBuilder(
                        intent = analysis,
                        schema = K_i.schema,
                        filters = ExtractFilters(analysis.entities, K_i.schema),
                        query_type = IF analysis.requires_aggregation
                                     THEN AGGREGATION
                                     ELSE SEARCH,
                        search_text = QueryRewrite(q_raw, target=K_i.type)
                    )
                    queries.APPEND((K_i, typed_query))
                trace.Log(CONSTRUCT, queries)
                state ← EXECUTE

            CASE EXECUTE:
                results ← []
                PARALLEL FOR EACH (K_i, tq) IN queries:
                    WITH Timeout(remaining_budget(deadline)):
                        r ← K_i.Execute(tq)
                        results.APPEND({
                            "collection": K_i.id,
                            "query": tq,
                            "documents": r.documents,
                            "aggregations": r.aggregations,
                            "latency_ms": r.latency
                        })
                trace.Log(EXECUTE, results)
                state ← EVALUATE

            CASE EVALUATE:
                // Assess sufficiency
                all_docs ← FlatMap(r.documents FOR r IN results)
                sufficiency ← ComputeSufficiency(all_docs, analysis.intents)
                trace.Log(EVALUATE, {"sufficiency": sufficiency})

                IF sufficiency ≥ config.θ_sufficiency:
                    state ← SYNTHESIZE
                ELSE IF repair_count < config.max_repair_iterations:
                    state ← REPAIR
                ELSE:
                    // Graceful degradation: synthesize with partial results
                    LOG(severity="WARN", event="partial_results")
                    state ← SYNTHESIZE

            CASE REPAIR:
                repair_count += 1
                repair_strategy ← DetermineRepairStrategy(
                    analysis, results, sufficiency
                )
                // Strategies:
                // 1. BROADEN_QUERY: relax filters, expand terms
                // 2. ALTERNATE_COLLECTION: try next-ranked collection
                // 3. DECOMPOSE: break remaining gaps into sub-queries
                // 4. TOOL_INVOKE: call external API for missing data

                SWITCH repair_strategy:
                    CASE BROADEN_QUERY:
                        analysis ← RelaxFilters(analysis)
                        state ← CONSTRUCT
                    CASE ALTERNATE_COLLECTION:
                        selected ← NextRankedCollections(scored_collections)
                        state ← CONSTRUCT
                    CASE DECOMPOSE:
                        gap_queries ← DecomposeGaps(analysis, results)
                        // Execute gap queries
                        FOR EACH gq IN gap_queries:
                            gap_result ← Execute(gq)
                            results.APPEND(gap_result)
                        state ← EVALUATE
                    CASE TOOL_INVOKE:
                        tool_result ← tools.Invoke(
                            analysis.suggested_tool,
                            params = analysis.tool_params
                        )
                        results.APPEND(tool_result)
                        state ← EVALUATE

                trace.Log(REPAIR, repair_strategy)

            CASE SYNTHESIZE:
                context ← CompileSynthesisContext(
                    original_query = q_raw,
                    analysis = analysis,
                    results = results,
                    session = C_session,
                    token_budget = config.token_budget - token_consumed
                )
                response ← LLMSynthesize(
                    prompt = COMPILED_SYNTHESIS_PROMPT,
                    context = context,
                    output_schema = TypedResponse {
                        answer: str,
                        citations: List[Citation],
                        confidence: float,
                        caveats: List[str]
                    }
                )
                trace.Log(SYNTHESIZE, response)
                state ← COMMIT

            CASE COMMIT:
                response.trace ← trace
                response.provenance ← {
                    "collections_queried": [r.collection FOR r IN results],
                    "repair_iterations": repair_count,
                    "total_latency_ms": Now() - trace.start_time,
                    "token_consumed": token_consumed,
                    "sufficiency_final": sufficiency
                }

    // Timeout fallback
    IF state ≠ COMMIT:
        response ← GracefulDegradationResponse(
            partial_results = results,
            reason = "latency_budget_exceeded"
        )

    RETURN response, trace
────────────────────────────────────────────────────────────────
```

### 5.5 Contextual Awareness: Session Memory Integration

The query agent maintains **working memory** during execution and reads from **session memory** for conversational continuity:

$$
\mathcal{C}_{\text{effective}}(q_t) = q_t \oplus \text{Compress}(\{(q_{t-i}, r_{t-i})\}_{i=1}^{w})
$$

where $w$ is the session window size and $\text{Compress}$ is a summarization function that extracts entity bindings, resolved preferences, and established constraints from prior turns. This enables follow-up query resolution without re-specifying context.

---

## 6. Unified Pipeline: Query Compilation Stack

### 6.1 Architecture Diagram (Textual)

```
┌─────────────────────────────────────────────────────────┐
│                   USER QUERY (q_raw)                    │
└───────────────────────┬─────────────────────────────────┘
                        │
                        ▼
┌─────────────────────────────────────────────────────────┐
│  LAYER 1: SESSION CONTEXT INJECTION                     │
│  • Coreference resolution from session memory           │
│  • Entity binding from conversation history             │
│  • Constraint inheritance from prior turns              │
└───────────────────────┬─────────────────────────────────┘
                        │
                        ▼
┌─────────────────────────────────────────────────────────┐
│  LAYER 2: COMPLEXITY CLASSIFIER                         │
│  • Single-intent → Direct Rewrite path                  │
│  • Multi-intent → Decomposition path                    │
│  • Ambiguous → Clarification request (interrupt)        │
└──────┬────────────────┬──────────────────┬──────────────┘
       │                │                  │
       ▼                ▼                  ▼
┌──────────┐   ┌────────────────┐   ┌──────────────┐
│ REWRITE  │   │  DECOMPOSE     │   │ CLARIFY      │
│          │   │  (DAG Build)   │   │ (User Loop)  │
└────┬─────┘   └───────┬────────┘   └──────────────┘
     │                 │
     ▼                 ▼
┌─────────────────────────────────────────────────────────┐
│  LAYER 3: QUERY EXPANSION (per sub-query)               │
│  • Generative paraphrases with drift control            │
│  • HyDE for complex information needs                   │
│  • Ontology expansion for domain coverage               │
│  • MMR selection for diversity                          │
└───────────────────────┬─────────────────────────────────┘
                        │
                        ▼
┌─────────────────────────────────────────────────────────┐
│  LAYER 4: RETRIEVAL ROUTING                             │
│  • Schema-aware collection selection                    │
│  • Query type determination (vector/keyword/graph/API)  │
│  • Dynamic filter construction                          │
│  • Parallel execution with deadline propagation         │
└───────────────────────┬─────────────────────────────────┘
                        │
                        ▼
┌─────────────────────────────────────────────────────────┐
│  LAYER 5: SUFFICIENCY EVALUATION & REPAIR LOOP          │
│  • Coverage verification against intent set             │
│  • Repair: broaden, reroute, decompose gaps, tool call  │
│  • Bounded iterations with graceful degradation         │
└───────────────────────┬─────────────────────────────────┘
                        │
                        ▼
┌─────────────────────────────────────────────────────────┐
│  LAYER 6: SYNTHESIS & COMMIT                            │
│  • Cross-sub-query aggregation                          │
│  • Citation-linked response generation                  │
│  • Confidence scoring and caveat annotation             │
│  • Full provenance trace emission                       │
└─────────────────────────────────────────────────────────┘
```

### 6.2 Token Budget Allocation Model

Define the total token budget $B_{\text{total}}$ for the context window. The query compilation stack allocates:

| Component | Budget Fraction | Justification |
|---|---|---|
| System prompt + role policy | $0.08 \cdot B_{\text{total}}$ | Fixed overhead, amortized across calls |
| Session memory (compressed) | $0.10 \cdot B_{\text{total}}$ | Bounded by window size $w$ and compression ratio |
| Rewritten/expanded queries | $0.05 \cdot B_{\text{total}}$ | Queries are short; budget is for provenance metadata |
| Retrieved documents | $0.55 \cdot B_{\text{total}}$ | Primary evidence payload |
| Synthesis instructions | $0.07 \cdot B_{\text{total}}$ | Structured output schema, citation format |
| Generation headroom | $0.15 \cdot B_{\text{total}}$ | Reserved for response tokens |

**Hard constraint:** If retrieved documents exceed their allocation, apply **authority-ranked truncation**: score each document by $\text{Authority}(d) \cdot \text{Relevance}(d, q^{*}) \cdot \text{Freshness}(d)$ and drop lowest-ranked documents until within budget.

---

## 7. Production Reliability Engineering

### 7.1 Failure Modes and Mitigations

| Failure Mode | Detection Mechanism | Mitigation |
|---|---|---|
| **Rewrite semantic drift** | Embedding similarity gate | Fallback to original query |
| **Expansion over-generation** | $n > n_{\max}$ or total tokens exceed budget | MMR truncation with drift-priority ordering |
| **Decomposition cycle** | DAG cycle detection | Hard assertion; reject and re-decompose |
| **Collection routing miss** | Zero results from all selected collections | Expand to all collections with relaxed filters |
| **Agent infinite loop** | Repair iteration counter | Hard bound + graceful degradation response |
| **Latency budget exceeded** | Deadline propagation via context | Return partial results with explicit caveat |
| **LLM hallucination in rewrite** | Entity coverage check | Reject rewrite if entities are missing |
| **Stale session context** | TTL on session memory entries | Expire entries older than $T_{\text{session\_ttl}}$ |

### 7.2 Observability Requirements

Every query augmentation operation emits structured traces:

```protobuf
message QueryAugmentationTrace {
    string trace_id = 1;
    string original_query = 2;
    QueryRewriteEvent rewrite = 3;
    repeated QueryExpansionEvent expansions = 4;
    repeated QueryDecompositionEvent decompositions = 5;
    repeated RetrievalRoutingEvent routings = 6;
    repeated RepairEvent repairs = 7;
    SufficiencyScore final_sufficiency = 8;
    int64 total_latency_ms = 9;
    int32 total_tokens_consumed = 10;
    string model_version = 11;
}
```

These traces feed into the **continuous evaluation infrastructure**: failed traces are converted into regression test cases, sufficiency scores below threshold trigger automated review, and drift patterns across queries are monitored for systematic rewriter degradation.

### 7.3 Idempotency and Determinism

For the same $(q_{\text{raw}}, \mathcal{C}_{\text{session}}, \text{schema\_state})$ tuple:

- **Rewriting** with temperature $T = 0$ and fixed model version produces deterministic output
- **Expansion** uses seeded random selection when MMR scores are tied
- **Decomposition** DAG structure is cached and reused for identical queries within a session
- **Routing** scores are computed from deterministic schema metadata and cached yield statistics

### 7.4 Cost Optimization

$$
\text{Cost}(\mathcal{A}_q) = \sum_{i=1}^{n_{\text{LLM\_calls}}} \text{TokenCost}(c_i) + \sum_{j=1}^{n_{\text{retrievals}}} \text{RetrievalCost}(r_j)
$$

**Optimization strategies:**

1. **Tiered rewriting:** Simple queries (classified by complexity detector) skip LLM rewriting entirely; only compound/ambiguous queries invoke the full stack
2. **Cached expansions:** Frequent query patterns are cached with TTL, avoiding redundant LLM expansion calls
3. **Lazy decomposition:** Decomposition is triggered only when complexity classifier outputs COMPOUND or MULTI_HOP
4. **Early termination:** If the first retrieval pass achieves $\text{Sufficiency} \geq 0.95$, skip expansion and additional collection routing entirely

---

## 8. Mathematical Summary: End-to-End Query Augmentation as Optimization

The complete query augmentation pipeline can be expressed as a constrained optimization:

$$
q^{*} = \arg\max_{q' \in \mathcal{T}(q_{\text{raw}})} \text{Recall}@k(q') \cdot \text{Precision}@k(q')
$$

subject to:

$$
\text{sim}(\mathbf{e}(q'), \mathbf{e}(q_{\text{raw}})) \geq \theta_{\text{preserve}}
$$

$$
\text{Latency}(\mathcal{T}) \leq L_{\max}
$$

$$
\text{TokenCost}(\mathcal{T}) \leq C_{\max}
$$

$$
\text{RepairIterations} \leq R_{\max}
$$

where $\mathcal{T}$ is the set of all query transformations reachable through the pipeline's rewrite, expand, decompose, and route operations.

---

## 9. Conclusion

Query augmentation in production agentic systems is not a preprocessing convenience — it is the **first typed compilation stage** in a deterministic context engineering stack. The techniques of rewriting, expansion, decomposition, and agentic query orchestration form a layered, composable architecture where each layer has explicit quality gates, bounded failure modes, provenance tracking, and measurable optimization criteria. The system must treat every raw user query as an unreliable signal that requires mechanical transformation, verification, and routing before it can be admitted into the retrieval and synthesis pipeline. Without this discipline, the entire downstream agentic execution operates on a foundation of uncontrolled noise.

---
